# 39 — Tiny Decoder Readiness Capstone

**Master syllabus coverage:** V2 Part V capstone and bridge to Part VI

## Why this matters

This capstone integrates the entire prerequisite path into one falsifiable result: a transparent decoder that passes invariants, overfits a fixed batch, and generates autoregressively.

## Learning objectives

- Assemble a complete character-level decoder from explicit components.
- Verify shape, causal, finite-gradient, and weight-tying contracts.
- Intentionally overfit one deterministic batch.
- Generate autoregressively and document architecture, experiment state, and limitations.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## Capstone contract

Build a tiny character-level decoder without `nn.Transformer`, overfit one fixed batch,
generate text, and produce architecture/verification evidence. This is a proof of
understanding, not the production SLM. Keep it deliberately small and deterministic.

Success criteria in this notebook:

- input `[B,T]` → logits `[B,T,V]`;
- every prefix logit is independent of future input tokens;
- all trainable parameters receive finite gradients;
- one fixed batch loss drops far below the uniform baseline `log(V)`;
- greedy generation uses only the available context window.


In [ ]:
import math
import random
from dataclasses import asdict, dataclass
import numpy as np
import torch
import torch.nn.functional as F

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.set_num_threads(1)

text = ("timing makes the joke. " * 20).strip()
vocabulary = sorted(set(text))
stoi = {character: index for index, character in enumerate(vocabulary)}
itos = dict(enumerate(vocabulary))
encoded = torch.tensor([stoi[character] for character in text], dtype=torch.long)
print("characters:", len(text), "vocab:", len(vocabulary), vocabulary)


## 1. Configuration and model

Configuration is immutable, serializable experiment state. The model uses learned absolute
positions, pre-norm blocks, causal SDPA, GELU MLPs, final LayerNorm, and tied embeddings.


In [ ]:
@dataclass(frozen=True)
class Config:
    vocab_size: int
    context: int = 16
    width: int = 32
    heads: int = 4
    layers: int = 2

class Attention(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.width % config.heads == 0
        self.heads = config.heads
        self.head_dim = config.width // config.heads
        self.qkv = torch.nn.Linear(config.width, 3 * config.width)
        self.out = torch.nn.Linear(config.width, config.width)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split(z): return z.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split, (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(config.width)
        self.attention = Attention(config)
        self.norm2 = torch.nn.LayerNorm(config.width)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(config.width, 4 * config.width), torch.nn.GELU(),
            torch.nn.Linear(4 * config.width, config.width),
        )
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class TinyDecoder(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = torch.nn.Embedding(config.vocab_size, config.width)
        self.position_embedding = torch.nn.Embedding(config.context, config.width)
        self.blocks = torch.nn.ModuleList([Block(config) for _ in range(config.layers)])
        self.final_norm = torch.nn.LayerNorm(config.width)
        self.lm_head = torch.nn.Linear(config.width, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
    def forward(self, ids, targets=None):
        B, T = ids.shape
        if T > self.config.context:
            raise ValueError("input exceeds configured context")
        positions = torch.arange(T, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None]
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), targets.reshape(-1)
        )
        return logits, loss


## 2. Build one deterministic batch

Overfitting one batch is a diagnostic: it proves the graph can represent and optimize
these examples. It does not prove generalization. Use varied start positions so the batch
contains multiple local contexts while remaining exactly repeatable.


In [ ]:
config = Config(vocab_size=len(vocabulary))
starts = torch.tensor([0, 3, 7, 11, 17, 23, 31, 41])
batch_x = torch.stack([encoded[start:start + config.context] for start in starts])
batch_y = torch.stack([encoded[start + 1:start + config.context + 1] for start in starts])
model = TinyDecoder(config)
logits, initial_loss = model(batch_x, batch_y)
print("config:", asdict(config))
print("input/target/logits:", batch_x.shape, batch_y.shape, logits.shape)
print("parameters:", sum(p.numel() for p in model.parameters()))
print("initial loss / uniform baseline:", initial_loss.item(), math.log(config.vocab_size))
assert logits.shape == (*batch_x.shape, config.vocab_size)


## 3. Verify the graph before training

Check weight identity, end-to-end causality, finite forward values, and gradient coverage.
The causality test changes only the suffix and compares prefix logits. Run in eval mode so
no stochastic layer could confound the result.


In [ ]:
assert model.token_embedding.weight is model.lm_head.weight
model.eval()
prefix_length = 8
changed = batch_x[:1].clone()
changed[:, prefix_length:] = torch.flip(changed[:, prefix_length:], dims=(1,))
with torch.inference_mode():
    original_logits, _ = model(batch_x[:1])
    changed_logits, _ = model(changed)
torch.testing.assert_close(original_logits[:, :prefix_length], changed_logits[:, :prefix_length],
                           atol=1e-5, rtol=1e-5)

model.train()
_, check_loss = model(batch_x, batch_y)
check_loss.backward()
missing = [name for name, parameter in model.named_parameters() if parameter.grad is None]
nonfinite = [name for name, parameter in model.named_parameters()
             if parameter.grad is not None and not torch.isfinite(parameter.grad).all()]
assert not missing and not nonfinite
model.zero_grad(set_to_none=True)
print("causality, weight tying, gradient coverage, and finiteness passed")


## 4. Overfit the fixed batch

AdamW is used without weight decay because this is a memorization diagnostic. Record loss
periodically, assert a strong decrease, and stop if values become non-finite. In a real run,
train and validation data, schedules, clipping, checkpoints, and experiment metadata return.


In [ ]:
torch.manual_seed(seed)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.0)
losses = []
for step in range(301):
    _, loss = model(batch_x, batch_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(loss) and torch.isfinite(gradient_norm)
    optimizer.step()
    losses.append(loss.item())
    if step % 75 == 0:
        print(f"step={step:3}, loss={loss.item():.4f}, grad_norm={gradient_norm.item():.3f}")

assert losses[-1] < 0.08 and losses[-1] < losses[0] * 0.05


## 5. Autoregressive generation

Training computes all positions in parallel; generation produces one token at a time.
At each step crop to the last `context` IDs, take final-position logits, choose one next
ID, append, and repeat. Greedy decoding is deterministic and useful for this diagnostic.


In [ ]:
@torch.inference_mode()
def generate(model, start_ids, new_tokens):
    model.eval()
    ids = start_ids.clone()
    for _ in range(new_tokens):
        context_ids = ids[:, -model.config.context:]
        logits, _ = model(context_ids)
        next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
        ids = torch.cat((ids, next_id), dim=1)
    return ids

prompt = "timing"
prompt_ids = torch.tensor([[stoi[character] for character in prompt]])
generated_ids = generate(model, prompt_ids, 50)[0].tolist()
generated_text = "".join(itos[index] for index in generated_ids)
print(generated_text)
assert generated_text.startswith(prompt)


## 6. Architecture and experiment report

Fill this out in your own words before leaving the notebook:

| Item | This run |
| --- | --- |
| Objective | character next-token cross-entropy |
| Shapes | input `[8,16]`, logits `[8,16,V]` |
| Architecture | learned token/position embedding, 2 pre-norm decoder blocks, tied head |
| Verification | shapes, causality, finite values/gradients, weight identity, one-batch overfit |
| Reproducibility | seed 42, fixed in-notebook corpus/config/command; add Git commit and device in a real run |
| Limitation | memorizes a repeated tiny corpus; no evidence of general language ability |

The production `brain/` project begins by translating this understood baseline into tested,
reusable modules and a real experiment/data pipeline. Modern components come after baseline
equivalence, so every change has a measured reason.


## Failure modes to recognize

- **Cannot overfit one batch:** suspect target shift, causal mask, gradients, capacity, or optimizer before collecting more data.
- **Prefix changes with suffix:** future-token leakage invalidates training loss.
- **Training/generation context mismatch:** positions or crop policy differ.
- **Memorization called generalization:** the diagnostic result is overstated.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Reimplement the attention calculation manually and prove output/gradient equivalence with SDPA.
2. Add stochastic temperature/top-k sampling and separate model probabilities from sampling policy.
3. Save and resume at step 150, then verify the resumed loss trajectory is identical to uninterrupted training.
4. Write a one-page readiness report answering every final gate in `SYLLABUS.md` with notebook evidence.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can rebuild the decoder without copying, explain every shape and parameter, pass all invariants, overfit one batch, and state honestly what the result does not prove.

**Next:** Move to `brain/` milestone B1 — Train MiniGPT as a reusable, reproducible project.
